# ASL Alphabet Sign Language Classification — Starter Notebook

This notebook explores the **26K ASL Sign Language Images Dataset (A-Z)** (SignAlphaSet, CC BY 4.0)
and trains a baseline CNN to classify hand signs A–Z.

**What this notebook covers:**
- Dataset overview & class distribution
- Sample image visualization
- A simple CNN baseline (Keras)
- Training curves & evaluation

Dataset source: SignAlphaSet, Mendeley Data, DOI 10.17632/8fmvr9m98w.3 (CC BY 4.0)


## 1. Setup & Imports

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator

print("TensorFlow version:", tf.__version__)


## 2. Dataset Path & Overview

Update `DATA_DIR` to match your dataset's path under `/kaggle/input/`.

In [ ]:
# Update this path to match your dataset slug on Kaggle
DATA_DIR = "/kaggle/input/26k-asl-sign-language-images-dataset-a-z"

classes = sorted(os.listdir(DATA_DIR))
print("Number of classes:", len(classes))
print("Classes:", classes)


## 3. Class Distribution

In [ ]:
counts = {}
for c in classes:
    folder = os.path.join(DATA_DIR, c)
    if os.path.isdir(folder):
        counts[c] = len(os.listdir(folder))

dist_df = pd.DataFrame(list(counts.items()), columns=["letter", "image_count"]).sort_values("letter")
display(dist_df)

plt.figure(figsize=(14,5))
plt.bar(dist_df["letter"], dist_df["image_count"], color="steelblue")
plt.title("Image Count per ASL Letter Class")
plt.xlabel("Letter")
plt.ylabel("Number of Images")
plt.show()


## 4. Sample Images

In [ ]:
import random
from PIL import Image

sample_letters = random.sample(classes, min(10, len(classes)))
fig, axes = plt.subplots(2, 5, figsize=(15,6))
for ax, letter in zip(axes.flatten(), sample_letters):
    folder = os.path.join(DATA_DIR, letter)
    img_name = random.choice(os.listdir(folder))
    img = Image.open(os.path.join(folder, img_name))
    ax.imshow(img)
    ax.set_title(letter)
    ax.axis("off")
plt.suptitle("Sample ASL Hand Signs")
plt.tight_layout()
plt.show()


## 5. Data Generators (Train/Validation Split)

In [ ]:
IMG_SIZE = (128, 128)
BATCH_SIZE = 32

datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2
)

train_gen = datagen.flow_from_directory(
    DATA_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    subset="training",
    seed=42
)

val_gen = datagen.flow_from_directory(
    DATA_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    subset="validation",
    seed=42
)

num_classes = train_gen.num_classes
print("Number of classes detected:", num_classes)


## 6. Baseline CNN Model

In [ ]:
model = models.Sequential([
    layers.Input(shape=(IMG_SIZE[0], IMG_SIZE[1], 3)),
    layers.Conv2D(32, (3,3), activation="relu"),
    layers.MaxPooling2D(2,2),
    layers.Conv2D(64, (3,3), activation="relu"),
    layers.MaxPooling2D(2,2),
    layers.Conv2D(128, (3,3), activation="relu"),
    layers.MaxPooling2D(2,2),
    layers.Flatten(),
    layers.Dense(256, activation="relu"),
    layers.Dropout(0.4),
    layers.Dense(num_classes, activation="softmax")
])

model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()


## 7. Train the Model

In [ ]:
EPOCHS = 10

history = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=EPOCHS
)


## 8. Training Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14,5))

axes[0].plot(history.history["accuracy"], label="Train Accuracy")
axes[0].plot(history.history["val_accuracy"], label="Val Accuracy")
axes[0].set_title("Accuracy over Epochs")
axes[0].set_xlabel("Epoch")
axes[0].legend()

axes[1].plot(history.history["loss"], label="Train Loss")
axes[1].plot(history.history["val_loss"], label="Val Loss")
axes[1].set_title("Loss over Epochs")
axes[1].set_xlabel("Epoch")
axes[1].legend()

plt.tight_layout()
plt.show()


## 9. Evaluate & Confusion Matrix

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

val_gen.reset()
preds = model.predict(val_gen)
y_pred = np.argmax(preds, axis=1)
y_true = val_gen.classes

print(classification_report(y_true, y_pred, target_names=list(train_gen.class_indices.keys())))

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(14,12))
sns.heatmap(cm, annot=False, cmap="Blues",
            xticklabels=list(train_gen.class_indices.keys()),
            yticklabels=list(train_gen.class_indices.keys()))
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.show()


## 10. Conclusion & Next Steps

This baseline CNN gives a starting point for ASL letter classification. Ideas to improve:
- Use transfer learning (MobileNetV2, EfficientNet, ResNet50)
- Add data augmentation (rotation, zoom, brightness shifts)
- Increase image resolution
- Try an LSTM-based approach on sequential/video data for dynamic gestures

**Dataset credit:** SignAlphaSet by Garg, Kasar, Kashyap, Vats, Sharma & Hange —
Bharati Vidyapeeth Deemed University College of Engineering, Pune.
Mendeley Data, DOI 10.17632/8fmvr9m98w.3 — CC BY 4.0.
